In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Silver
# Notebook        : Silver_employees
# Source          : employees.csv
# Target          : procurement.silver.silver_employees
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads Cleaned employees master data into the Silver layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Read employee data from the Bronze layer, apply data cleansing,standardization, and business rules to create a trusted Silver Delta table.
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim,coalesce,initcap,lower)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import try_to_timestamp

In [0]:
# ============================================================
# Read Bronze Employee Table
# ============================================================

bronze_employees_df = read_delta(BRONZE_EMPLOYEES)

preview(bronze_employees_df,"Bronze employees")

In [0]:
# ============================================================
# Identify invalid employee records
# NULL & Blank employee id ,null department id and null hire date
# ============================================================
invalid_employees = bronze_employees_df.filter(
    col("employee_id").isNull()
    |(trim(col("employee_id")) == "")
    |col("department_id").isNull()
    |col("hire_date").isNull()
    )

print(f"Invalid Employee Records:{invalid_employees.count()}")
display(invalid_employees)

In [0]:
# ============================================================
# Add Audit Metadata
# ============================================================

invalid_employees = (invalid_employees.withColumn("audit_timestamp",current_timestamp())
    .withColumn("source_table",lit("Employees"))
    .withColumn("pipeline_layer",lit("Silver"))
    .withColumn("issue_type",lit("Invalid Record")))

display(invalid_employees)

In [0]:
# ============================================================
# Write Invalid Employee Audit Table
# ============================================================

if invalid_employees.count() > 0:
    write_delta(invalid_employees,AUDIT_INVALID_EMPLOYEES,mode="overwrite")
    print("Invalid employee records written.")
else:
    print("No invalid employee records found.")

In [0]:
# ============================================================
# Remove Invalid Records
# ============================================================

silver_employees_df = bronze_employees_df.filter(col("employee_id").isNotNull()

    & (trim(col("employee_id")) != "")

    & col("department_id").isNotNull()

    & col("hire_date").isNotNull()

)

display(silver_employees_df)

In [0]:
# ============================================================
# Remove Duplicate Employees
# ============================================================

window_spec = Window.partitionBy("employee_id").orderBy("employee_id")

silver_employees_df = (silver_employees_df.withColumn("row_num",row_number().over(window_spec))
                      .filter(col("row_num") == 1).drop("row_num"))
preview(silver_employees_df,"Silver Employees")


In [0]:
# ============================================================
# Apply Business Transformations
# ============================================================
from pyspark.sql.functions import coalesce, to_date, trim, col
from pyspark.sql.functions import expr

silver_employees_df = (
    silver_employees_df
    .withColumn("employee_name", initcap(trim(col("employee_name"))))
    .withColumn("department_id", trim(col("department_id")))
    .withColumn("job_title",when(col("job_title").isNull(), "Unknown").otherwise(initcap(trim(col("job_title"))))
    )
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("hire_date",coalesce(
            try_to_timestamp(trim(col("hire_date")), lit("dd-MM-yyyy")).cast("date"),
            try_to_timestamp(trim(col("hire_date")), lit("dd-MMM-yy")).cast("date"),
            try_to_timestamp(trim(col("hire_date")), lit("dd MMMM yyyy")).cast("date")
        )
    )
)

display(silver_employees_df)

In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

silver_employees_df = (silver_employees_df
                         .withColumn("silver_load_timestamp", current_timestamp())
                         .withColumn("pipeline_layer", lit("Silver"))
)

In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

write_delta(df=silver_employees_df,table_name=SILVER_EMPLOYEES)

In [0]:
# ============================================================
# Validate Summary
# ============================================================

print("=" * 60)
print("Silver Employee Load Completed Successfully")
print("=" * 60)

print(f"Bronze Records : {bronze_employees_df.count()}")
print(f"Silver Records : {silver_employees_df.count()}")
print(f"Duplicates Removed : {bronze_employees_df.count() - silver_employees_df.count()}")